# Prototype 2 — Multi-Item Segmentation + Tagging

Validates the bulk-add pipeline end-to-end:
1. Detect N items via Gemini 2.5 spatial segmentation
2. Crop each bbox
3. Remove background per crop (rembg)
4. Categorize each cutout
5. Render N-card review grid (mirrors what `BulkAdd.tsx` will show)
6. Provider comparison + metrics table

Inputs: `inputs/20260524_174630.jpg` + `inputs/20260524_174709.jpg`
Outputs saved to `../outputs/`.

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from PIL import Image

load_dotenv(Path('..') / '.env')
sys.path.insert(0, str(Path('..')))

from lib.client import VISION_MODEL
from lib.images import resize_image, crop_bbox
from lib.detect_items import detect_items
from lib.categorize import categorize_item
from lib.remove_background import remove_background, REMBG_AVAILABLE
from lib.metrics import MetricsRecorder

metrics = MetricsRecorder()
print(f'VISION_MODEL: {VISION_MODEL}')
print(f'API key set: {bool(os.getenv("GEMINI_API_KEY"))}')
print(f'rembg available: {REMBG_AVAILABLE}')
print(f'fal.ai: {"enabled" if os.getenv("FAL_KEY") else "disabled (set FAL_KEY in .env)"}')
Path('../outputs').mkdir(exist_ok=True)

## Step 1 — Load input photos

In [ ]:
INPUT_DIR = Path('../inputs')
photos = [
    INPUT_DIR / '20260524_174630.jpg',
    INPUT_DIR / '20260524_174709.jpg',
]

imgs = [resize_image(Image.open(p)) for p in photos]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, img, p in zip(axes, imgs, photos):
    ax.imshow(img)
    ax.set_title(p.name)
    ax.axis('off')
plt.tight_layout()
plt.show()

img = imgs[0]  # primary photo for the main run
print(f'Working with: {photos[0].name} ({img.width}×{img.height})')

## Step 2 — Detect items (Gemini 2.5 spatial)

Returns `[{box_2d: [ymin, xmin, ymax, xmax], label: str}, ...]` in normalized [0, 1000] coordinates.

In [ ]:
print(f'Detecting items with {VISION_MODEL}...')
with metrics.record('gemini-2.5-flash', 'none') as run:
    detections = detect_items(img)
    run.items_found = len(detections)
    run.cost_usd_est = 0.002

print(f'Found {len(detections)} item(s):')
for i, d in enumerate(detections):
    print(f'  {i+1}. {d["label"]}  box_2d={d["box_2d"]}')

## Step 3 — Visualize bounding boxes

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(img)
colors = plt.cm.Set1.colors

for i, d in enumerate(detections):
    ymin, xmin, ymax, xmax = d['box_2d']
    h, w = img.height, img.width
    x0 = xmin / 1000 * w
    y0 = ymin / 1000 * h
    bw = (xmax - xmin) / 1000 * w
    bh = (ymax - ymin) / 1000 * h
    color = colors[i % len(colors)]
    rect = patches.Rectangle((x0, y0), bw, bh, linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x0, y0 - 5, d['label'], color=color, fontsize=9, fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, pad=2))

ax.set_title(f'Detected {len(detections)} items (Gemini 2.5 spatial)')
ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/02_detections.png', dpi=150)
plt.show()
print('Saved → outputs/02_detections.png')

## Step 4 — Crop each bbox

In [ ]:
crops = [crop_bbox(img, d['box_2d']) for d in detections]

cols = min(len(crops), 5)
rows = max(1, (len(crops) + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 4))
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, (crop, d) in enumerate(zip(crops, detections)):
    axes_flat[i].imshow(crop)
    axes_flat[i].set_title(d['label'][:30], fontsize=8)
    axes_flat[i].axis('off')
for j in range(len(crops), len(axes_flat)):
    axes_flat[j].axis('off')

plt.suptitle('Cropped items', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/02_crops.png', dpi=150)
plt.show()

## Step 5 — Background removal per crop (rembg)

> Requires `pip install rembg onnxruntime`. First run downloads ~175MB ONNX model.

In [ ]:
if not REMBG_AVAILABLE:
    print('rembg not installed — skipping. Run: pip install rembg onnxruntime')
    cutouts = crops  # fall back to raw crops for downstream cells
else:
    print('Removing backgrounds (rembg)...')
    cutouts = []
    with metrics.record('gemini-2.5-flash', 'rembg') as run:
        for i, crop in enumerate(crops):
            print(f'  Processing crop {i+1}/{len(crops)}...')
            cutout = remove_background(crop, provider='rembg')
            cutouts.append(cutout)
        run.items_found = len(crops)
        run.cost_usd_est = 0.0  # rembg is free

    cols = min(len(cutouts), 5)
    rows = max(1, (len(cutouts) + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 4))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, (cutout, d) in enumerate(zip(cutouts, detections)):
        display = Image.new('RGBA', cutout.size, (220, 220, 220, 255))
        if cutout.mode == 'RGBA':
            display.paste(cutout, mask=cutout.split()[3])
        axes_flat[i].imshow(display)
        axes_flat[i].set_title(d['label'][:30], fontsize=8)
        axes_flat[i].axis('off')
    for j in range(len(cutouts), len(axes_flat)):
        axes_flat[j].axis('off')

    plt.suptitle('Cutouts (rembg)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/02_cutouts.png', dpi=150)
    plt.show()

## Step 6 — Categorize each cutout

In [ ]:
print('Categorizing each item...')
classifications = []
for i, cutout in enumerate(cutouts):
    print(f'  Item {i+1}/{len(cutouts)}: {detections[i]["label"]}...')
    cl = categorize_item(cutout.convert('RGB') if hasattr(cutout, 'convert') else cutout)
    classifications.append(cl)
    print(f'    → {cl.get("category")} / {cl.get("subcategory")} ({cl.get("primary_color")})')

## Step 7 — Render N-card review grid

Mirrors the visual shape of `BulkAdd.tsx` step 2.

In [ ]:
n = len(classifications)
cols = min(n, 3)
rows = max(1, (n + cols - 1) // cols)
fig = plt.figure(figsize=(cols * 5, rows * 7))
gs = gridspec.GridSpec(rows, cols, figure=fig, hspace=0.5, wspace=0.3)

for i, (cutout, d, cl) in enumerate(zip(cutouts, detections, classifications)):
    row, col = divmod(i, cols)
    ax = fig.add_subplot(gs[row, col])

    display = Image.new('RGBA', cutout.size, (245, 245, 245, 255))
    if cutout.mode == 'RGBA':
        display.paste(cutout, mask=cutout.split()[3])
    else:
        display = cutout

    ax.imshow(display)
    tags_text = (
        f"{cl.get('category', '—')} / {cl.get('subcategory', '—')}\n"
        f"Color: {cl.get('primary_color', '—')}\n"
        f"Material: {cl.get('material') or '—'}\n"
        f"Tags: {', '.join(cl.get('suggested_tags', []))}\n"
        f"\n{cl.get('ai_description', '')}"
    )
    ax.set_title(tags_text, fontsize=7, loc='left', wrap=True)
    ax.axis('off')

plt.suptitle(f'Review Grid — {n} detected items', fontsize=14, fontweight='bold')
plt.savefig('../outputs/02_review_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/02_review_grid.png')

## [Optional] Provider comparison — fal.ai BiRefNet

Set `FAL_KEY` in `.env` to enable. Compares background removal quality vs rembg.

In [ ]:
if not os.getenv('FAL_KEY'):
    print('FAL_KEY not set — skipping fal.ai comparison. Set FAL_KEY in .env to enable.')
else:
    print('Removing backgrounds with fal.ai BiRefNet v2...')
    fal_cutouts = []
    with metrics.record('gemini-2.5-flash', 'fal-birefnet') as run:
        for i, crop in enumerate(crops):
            print(f'  Processing crop {i+1}/{len(crops)}...')
            fal_cutout = remove_background(crop, provider='fal')
            fal_cutouts.append(fal_cutout)
        run.items_found = len(crops)
        run.cost_usd_est = len(crops) * 0.005  # rough estimate per crop

    cols = min(len(fal_cutouts), 5)
    rows = max(1, (len(fal_cutouts) + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 4))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for i, (cutout, d) in enumerate(zip(fal_cutouts, detections)):
        axes_flat[i].imshow(cutout)
        axes_flat[i].set_title(d['label'][:30], fontsize=8)
        axes_flat[i].axis('off')
    plt.suptitle('Cutouts (fal.ai BiRefNet v2)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('Set mask_quality manually in the metrics cell below after comparing visually.')

## Metrics table

Fill in `mask_quality` (1–5) after eyeballing the cutout outputs above.

In [ ]:
# Set mask_quality manually after visual inspection
# e.g. metrics.runs[0].mask_quality = 3  (rembg)
# e.g. metrics.runs[1].mask_quality = 5  (fal-birefnet, if run)

if metrics.runs:
    df = metrics.to_dataframe()
    print(df.to_string(index=False))
    df.to_csv('../outputs/02_metrics.csv', index=False)
    print('\nSaved → outputs/02_metrics.csv')
else:
    print('No metrics recorded yet — run the detection + BG-removal cells first.')